<a href="https://colab.research.google.com/github/rafaellopesdesa/nsbi-lhc-toolkit/blob/ml4hep_school_tutorial/workshops/ml4hep_tifr_colab/Exercise_9_Hybrid_NPE_NDE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Exercise 9 — Dual hybrid neural posterior and density estimation

The previous exercises were frequentist: learned densities and density ratios were assembled into a likelihood and then profiled. This exercise develops the Bayesian dual construction described in the hybrid NPE note:

1. train a conditional normalizing flow $q_\phi(\theta\mid x)$ by neural posterior estimation (NPE);
2. freeze it and train a matched-$x$ classifier for the residual posterior ratio;
3. show that the hybrid posterior is substantially closer to truth than the flow alone;
4. train the dual observation-space hNDE model $q_\eta(x)\,r_{\rm L}(x;\theta)$;
5. verify that the posterior-space and likelihood-space constructions agree;
6. use the dual model for posterior-predictive generation, absolute evidence, a nuisance-prior/auxiliary update, and selection integrals without further simulator calls.

All normalizing flows in this notebook are the **same rational-quadratic spline coupling flow** used in Exercises 4 and 5. The likelihood is analytically available for validation in this pedagogical model, but it is never supplied to either flow or classifier during training.


In [ ]:
## ============================================================================
# Google Colab setup — run me first. Safe to re-run; a no-op off Colab.
# ============================================================================
import os, sys
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/rafaellopesdesa/nsbi-lhc-toolkit.git"
BRANCH = "ml4hep_school_tutorial"
USE_DRIVE = True

def run(*args, env=None):
    subprocess.run([str(arg) for arg in args], check=True, env=env)

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        ROOT = Path("/content/drive/MyDrive/Colab Notebooks/ml4hep_tifr_colab")
    else:
        ROOT = Path("/content")
    ROOT.mkdir(parents=True, exist_ok=True)

    REPO_DIR = ROOT / "nsbi-lhc-toolkit"
    TUTORIAL_DIR = REPO_DIR / "workshops" / "ml4hep_tifr_colab"
    WORK_DIR = REPO_DIR / "workshops" / "ml4hep_tifr"
    if not (REPO_DIR / ".git").is_dir():
        clone_env = os.environ.copy()
        clone_env["GIT_LFS_SKIP_SMUDGE"] = "1"
        run(
            "git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
            "--branch", BRANCH, REPO_URL, REPO_DIR, env=clone_env,
        )
    else:
        run("git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL)
        run("git", "-C", REPO_DIR, "fetch", "origin", BRANCH)
        run("git", "-C", REPO_DIR, "checkout", BRANCH)
        run("git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH)
    run(
        "git", "-C", REPO_DIR, "sparse-checkout", "set",
        "src", "workshops/ml4hep_tifr_colab",
    )
    for import_dir in (REPO_DIR / "src", TUTORIAL_DIR):
        import_path = str(import_dir.resolve())
        if import_path not in sys.path:
            sys.path.insert(0, import_path)
    run(sys.executable, "-m", "pip", "install", "-q", "nflows")
    WORK_DIR.mkdir(parents=True, exist_ok=True)
    os.chdir(WORK_DIR)
else:
    # This covers execution from either the notebook directory or repo root.
    for candidate in [
        Path.cwd(),
        Path.cwd() / "workshops" / "ml4hep_tifr_colab",
    ]:
        if (candidate / "utils_hnpe.py").exists():
            sys.path.insert(0, str(candidate.resolve()))
            break

print("Working directory:", Path.cwd())


## A small Bayesian problem with a genuine nuisance parameter

We use

\[
  \theta=(\mu,\alpha),
\]

where $\mu$ is the parameter of interest and $\alpha$ is a detector-calibration nuisance. The three reconstructed observables are conditionally independent Gaussians,

\[
\begin{aligned}
  x_1 &\sim \mathcal N(\mu+0.8\alpha,\;0.95^2),\\
  x_2 &\sim \mathcal N(0.72\mu^2-0.4\alpha,\;0.38^2),\\
  x_3 &\sim \mathcal N(0.8\cos\mu+0.3\alpha,\;0.30^2).
\end{aligned}
\]

The quadratic and cosine responses make the posterior non-Gaussian and, for the chosen observation, bimodal in $\mu$. The scale-like nuisance shifts all three means coherently, echoing the detector response uncertainty in Exercise 8.

The simulation design is a factorized defensive mixture,

\[
  \rho(\mu)=0.9\,\mathcal N(0,1.5^2)+0.1\,\mathcal N(0,4^2),\qquad
  \rho(\alpha)=0.9\,\mathcal N(0,1^2)+0.1\,\mathcal N(0,3^2).
\]

Its broad components make later prior changes possible without creating a support mismatch. An optional auxiliary calibration is kept outside all neural training:

\[
  a\mid\alpha\sim\mathcal N(\alpha,\sigma_a^2).
\]


In [ ]:
import gc
import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from scipy.special import expit, logsumexp
from scipy.stats import norm

from utils_hnpe import (
    calibrate_ratio_classifier,
    ratio_classifier_logit,
    ratio_classifier_ratio,
    sample_spline_flow,
    spline_flow_log_prob,
    train_ratio_classifier,
    train_spline_flow,
)

SEED = 19092026
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# FAST_MODE is suitable for a first Colab pass. Set it to False for smoother
# final figures and more stable evidence/selection estimates.
FAST_MODE = True
LOAD_IF_AVAILABLE = True
RUN_TAG = "fast" if FAST_MODE else "full"
MODEL_DIR = Path("models_exercise9_hnpe_hnde_spline16_tail5") / RUN_TAG
MODEL_DIR.mkdir(parents=True, exist_ok=True)

if FAST_MODE:
    N_FLOW, N_RATIO, N_CALIBRATION = 35_000, 80_000, 16_000
    N_POSTERIOR, N_REFERENCE = 80_000, 35_000
    FLOW_EPOCHS, RATIO_EPOCHS = 12, 24
else:
    N_FLOW, N_RATIO, N_CALIBRATION = 120_000, 250_000, 50_000
    N_POSTERIOR, N_REFERENCE = 250_000, 100_000
    FLOW_EPOCHS, RATIO_EPOCHS = 28, 45

# Both q_phi(theta|x) and q_eta(x) use this same quadratic-spline family.
FLOW_MODEL_CONFIG = {
    "n_coupling_layers": 4,
    "hidden_features": 64,
    "hidden_layers": 2,
    "spline_num_bins": 16,
    "spline_tail_bound": 5.0,
    "dropout_probability": 0.0,
}
FLOW_TRAINING_CONFIG = {
    "batch_size": 1024,
    "n_epochs": FLOW_EPOCHS,
    "learning_rate": 2.0e-4,
    "lr_scheduler_factor": 0.3,
    "lr_scheduler_patience": 2,
    "min_learning_rate": 1.0e-6,
    "validation_fraction": 0.2,
    "patience": 5,
    "gradient_clip": 5.0,
}
RATIO_MODEL_CONFIG = {
    "hidden_features": 128,
    "hidden_layers": 3,
    "dropout_probability": 0.0,
}
RATIO_TRAINING_CONFIG = {
    "batch_size": 2048,
    "n_epochs": RATIO_EPOCHS,
    "learning_rate": 4.0e-4,
    "lr_scheduler_factor": 0.3,
    "lr_scheduler_patience": 2,
    "min_learning_rate": 1.0e-6,
    "validation_fraction": 0.2,
    "patience": 6,
    "gradient_clip": 5.0,
}

SIMULATOR_SIGMA = np.array([0.95, 0.38, 0.30])
X_OBS = np.array([0.40, 1.35, 0.12])
DEFENSIVE_EPSILON = 0.08


In [ ]:
def _mixture_logpdf(values, core_sigma, broad_sigma):
    values = np.asarray(values, dtype=float)
    return logsumexp(
        np.stack(
            [
                np.log(0.9) + norm.logpdf(values, 0.0, core_sigma),
                np.log(0.1) + norm.logpdf(values, 0.0, broad_sigma),
            ]
        ),
        axis=0,
    )


def design_mu_logpdf(mu):
    return _mixture_logpdf(mu, 1.5, 4.0)


def design_alpha_logpdf(alpha):
    return _mixture_logpdf(alpha, 1.0, 3.0)


def design_logpdf(theta):
    theta = np.atleast_2d(np.asarray(theta, dtype=float))
    return design_mu_logpdf(theta[:, 0]) + design_alpha_logpdf(theta[:, 1])


def _sample_mixture(n, core_sigma, broad_sigma, rng):
    broad = rng.random(int(n)) < 0.1
    sigma = np.where(broad, broad_sigma, core_sigma)
    return rng.normal(0.0, sigma)


def sample_design(n, rng):
    return np.column_stack(
        [
            _sample_mixture(n, 1.5, 4.0, rng),
            _sample_mixture(n, 1.0, 3.0, rng),
        ]
    ).astype(np.float32)


def simulator_mean(theta):
    theta = np.atleast_2d(np.asarray(theta, dtype=float))
    mu, alpha = theta[:, 0], theta[:, 1]
    return np.column_stack(
        [
            mu + 0.8 * alpha,
            0.72 * mu**2 - 0.4 * alpha,
            0.8 * np.cos(mu) + 0.3 * alpha,
        ]
    )


def simulate(theta, rng):
    mean = simulator_mean(theta)
    return (mean + rng.normal(size=mean.shape) * SIMULATOR_SIGMA).astype(
        np.float32
    )


def log_likelihood(x, theta):
    """Used only for truth validation; never passed to a network."""
    x = np.asarray(x, dtype=float)
    mean = simulator_mean(theta)
    return np.sum(norm.logpdf(x, loc=mean, scale=SIMULATOR_SIGMA), axis=1)


def auxiliary_loglikelihood(alpha, a_observed, sigma_a):
    return norm.logpdf(a_observed, loc=np.asarray(alpha), scale=sigma_a)


def normalized_weights_from_log(log_weights):
    log_weights = np.asarray(log_weights, dtype=float)
    weights = np.exp(log_weights - logsumexp(log_weights))
    return weights


def effective_sample_size(weights):
    weights = np.asarray(weights, dtype=float)
    weights = weights / weights.sum()
    return 1.0 / np.sum(weights**2)


print("Observed x:", X_OBS)
print("log p(x_obs | mu=1.2, alpha=-0.35) =", log_likelihood(X_OBS, [[1.2, -0.35]])[0])


## Independent simulation splits

Each learned object receives a separate simulator sample:

| split | use |
|---|---|
| $\mathcal D_\phi$ | train the conditional NPE $q_\phi(\theta\mid x)$ |
| $\mathcal D_{r_P}$ | train the posterior-residual classifier |
| $\mathcal D_\eta$ | train the observation flow $q_\eta(x)$ |
| $\mathcal D_{r_L}$ | train the likelihood-side classifier |
| $\mathcal D_{\rm cal}$ | independent affine logit calibration |

Calibration is a training operation, not a final validation. All comparisons with analytic truth below use new samples or deterministic quadrature. Keeping these roles separate prevents a classifier from being evaluated on the same fluctuations it learned.


## 1. Train the quadratic-spline NPE reference

Forward simulation provides pairs

\[
  \theta_i\sim\rho(\theta),\qquad x_i\sim p(x\mid\theta_i).
\]

Conditional maximum likelihood minimizes

\[
  -\frac1N\sum_i\log q_\phi(\theta_i\mid x_i),
\]

so $q_\phi$ approaches the design posterior $p_\rho(\theta\mid x)$. We intentionally keep this flow modest: the exercise needs a realistic residual error to correct, and the chosen observation has a subdominant second mode that an amortized finite-capacity flow can model imperfectly.


In [ ]:
rng = np.random.default_rng(SEED + 10)
theta_phi = sample_design(N_FLOW, rng)
x_phi = simulate(theta_phi, rng)

q_phi = train_spline_flow(
    theta_phi,
    context=x_phi,
    checkpoint=MODEL_DIR / "q_phi_conditional_posterior.pt",
    model_config=FLOW_MODEL_CONFIG,
    training_config=FLOW_TRAINING_CONFIG,
    device=device,
    seed=SEED + 11,
    load_if_available=LOAD_IF_AVAILABLE,
)

del theta_phi, x_phi
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


## 2. Add defensive support and train the posterior residual

A flow that assigns negligible probability to a posterior mode cannot be repaired by finite importance weights. We therefore use the normalized proposal

\[
  q_{\phi,\epsilon}(\theta\mid x)
  =(1-\epsilon)q_\phi(\theta\mid x)+\epsilon\rho(\theta),
  \qquad \epsilon=0.08.
\]

The small design component protects support; most proposals still come from the efficient NPE.

The classifier must learn

\[
  r_{\rm P}(\theta;x)
  =\frac{p_\rho(\theta\mid x)}{q_{\phi,\epsilon}(\theta\mid x)}.
\]

Positive pairs are ordinary simulator pairs. For every positive observation $x_i$, a negative parameter is sampled from the **actual frozen defensive proposal at the same $x_i$**:

\[
\begin{array}{lll}
  y=1:& \theta_i\sim\rho,&x_i\sim p(x\mid\theta_i),\\
  y=0:& \widetilde\theta_i\sim q_{\phi,\epsilon}(\theta\mid x_i),&\text{the same }x_i.
\end{array}
\]

With balanced classes, the optimal classifier logit is $\log r_{\rm P}$. Independently shuffling $\theta$ and $x$ would learn the ordinary likelihood-to-evidence ratio instead, so it would be the wrong negative construction.


In [ ]:
def sample_defensive_matched(context, rng):
    context = np.atleast_2d(np.asarray(context, dtype=np.float32))
    flow_draws = sample_spline_flow(q_phi, 1, context=context)[:, 0, :]
    design_draws = sample_design(len(context), rng)
    use_design = rng.random(len(context)) < DEFENSIVE_EPSILON
    draws = flow_draws.copy()
    draws[use_design] = design_draws[use_design]
    return draws.astype(np.float32)


def sample_defensive_fixed(x, n, rng):
    x = np.atleast_2d(np.asarray(x, dtype=np.float32))
    flow_draws = sample_spline_flow(q_phi, int(n), context=x)
    design_draws = sample_design(n, rng)
    use_design = rng.random(int(n)) < DEFENSIVE_EPSILON
    flow_draws[use_design] = design_draws[use_design]
    return flow_draws.astype(np.float32)


def defensive_logpdf(theta, x):
    theta = np.atleast_2d(np.asarray(theta, dtype=np.float32))
    x = np.atleast_2d(np.asarray(x, dtype=np.float32))
    if len(x) == 1:
        x = np.repeat(x, len(theta), axis=0)
    log_q_phi = spline_flow_log_prob(q_phi, theta, context=x)
    return np.logaddexp(
        np.log1p(-DEFENSIVE_EPSILON) + log_q_phi,
        np.log(DEFENSIVE_EPSILON) + design_logpdf(theta),
    )


rng = np.random.default_rng(SEED + 20)
theta_rp_positive = sample_design(N_RATIO, rng)
x_rp_matched = simulate(theta_rp_positive, rng)
theta_rp_negative = sample_defensive_matched(x_rp_matched, rng)

rp_positive = np.column_stack([theta_rp_positive, x_rp_matched])
rp_negative = np.column_stack([theta_rp_negative, x_rp_matched])
r_p = train_ratio_classifier(
    rp_positive,
    rp_negative,
    checkpoint=MODEL_DIR / "r_p_posterior_residual.pt",
    model_config=RATIO_MODEL_CONFIG,
    training_config=RATIO_TRAINING_CONFIG,
    device=device,
    seed=SEED + 21,
    load_if_available=LOAD_IF_AVAILABLE,
)

del theta_rp_positive, theta_rp_negative, x_rp_matched, rp_positive, rp_negative
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


### Calibrate the residual odds on an independent split

Density-ratio inference needs calibrated odds, not merely a large classifier AUC. We fit one affine transformation of the classifier logit on an independent balanced sample. This preserves ordering but corrects a global slope/intercept bias caused by finite training and regularization.

The reliability plot is a classifier-space diagnostic. The posterior comparison in the next section is the scientifically relevant validation.


In [ ]:
def reliability_data(pack, positive, negative, n_bins=12):
    values = np.concatenate([positive, negative])
    labels = np.concatenate([np.ones(len(positive)), np.zeros(len(negative))])
    probability = expit(ratio_classifier_logit(pack, values))
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    centers, observed, counts = [], [], []
    for low, high in zip(edges[:-1], edges[1:]):
        mask = (probability >= low) & (probability < high)
        if mask.sum() >= 25:
            centers.append(probability[mask].mean())
            observed.append(labels[mask].mean())
            counts.append(mask.sum())
    return np.asarray(centers), np.asarray(observed), np.asarray(counts)


rng = np.random.default_rng(SEED + 30)
theta_cal = sample_design(N_CALIBRATION, rng)
x_cal = simulate(theta_cal, rng)
theta_cal_negative = sample_defensive_matched(x_cal, rng)
rp_cal_positive = np.column_stack([theta_cal, x_cal])
rp_cal_negative = np.column_stack([theta_cal_negative, x_cal])

before_pack = dict(r_p)
before_pack["calibration_slope"] = 1.0
before_pack["calibration_intercept"] = 0.0
before = reliability_data(before_pack, rp_cal_positive, rp_cal_negative)
r_p = calibrate_ratio_classifier(r_p, rp_cal_positive, rp_cal_negative)
after = reliability_data(r_p, rp_cal_positive, rp_cal_negative)

fig, ax = plt.subplots(figsize=(5.2, 4.5))
ax.plot([0, 1], [0, 1], color="black", ls="--", lw=1, label="ideal")
ax.plot(before[0], before[1], "o-", label="raw classifier")
ax.plot(after[0], after[1], "o-", label="affine calibrated")
ax.set(xlabel="predicted class probability", ylabel="observed positive fraction")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

del theta_cal, theta_cal_negative, x_cal, rp_cal_positive, rp_cal_negative


## 3. Does hNPE correct the NPE?

For the fixed observation $x_o$, draw

\[
  \theta_j\sim q_{\phi,\epsilon}(\theta\mid x_o),\qquad
  u_j^{\rm P}=\widehat r_{\rm P}(\theta_j;x_o),\qquad
  w_j^{\rm P}=\frac{u_j^{\rm P}}{\sum_k u_k^{\rm P}}.
\]

We compare four objects:

- analytic truth, used only for validation;
- the uncorrected spline NPE $q_\phi$;
- the unweighted defensive proposal $q_{\phi,\epsilon}$;
- the ratio-corrected hNPE posterior.

A Jensen--Shannon (JS) distance is calculated from a common two-dimensional histogram. It is zero only for identical distributions. The effective sample size (ESS) reports the price paid for the correction.


In [ ]:
MU_GRID = np.linspace(-3.5, 3.5, 281)
ALPHA_GRID = np.linspace(-3.0, 3.0, 241)
MU_MESH, ALPHA_MESH = np.meshgrid(MU_GRID, ALPHA_GRID, indexing="ij")
THETA_GRID = np.column_stack([MU_MESH.ravel(), ALPHA_MESH.ravel()])

log_joint_truth = design_logpdf(THETA_GRID) + log_likelihood(X_OBS, THETA_GRID)
log_joint_truth = log_joint_truth.reshape(MU_MESH.shape)
truth_shift = float(log_joint_truth.max())
truth_unnormalized = np.exp(log_joint_truth - truth_shift)
truth_integral_scaled = np.trapezoid(
    np.trapezoid(truth_unnormalized, ALPHA_GRID, axis=1),
    MU_GRID,
)
LOG_EVIDENCE_TRUTH = truth_shift + np.log(truth_integral_scaled)
POSTERIOR_TRUTH = truth_unnormalized / truth_integral_scaled
TRUTH_MU = np.trapezoid(POSTERIOR_TRUTH, ALPHA_GRID, axis=1)
TRUTH_ALPHA = np.trapezoid(POSTERIOR_TRUTH, MU_GRID, axis=0)

rng = np.random.default_rng(SEED + 40)
x_context = X_OBS[None, :].astype(np.float32)
theta_npe = sample_spline_flow(q_phi, N_POSTERIOR, context=x_context)
theta_posterior = sample_defensive_fixed(X_OBS, N_POSTERIOR, rng)
rp_inputs = np.column_stack(
    [theta_posterior, np.repeat(x_context, len(theta_posterior), axis=0)]
)
log_rp = ratio_classifier_logit(r_p, rp_inputs)
weights_p = normalized_weights_from_log(log_rp)
print(
    f"hNPE ESS = {effective_sample_size(weights_p):,.0f} / {len(weights_p):,} "
    f"({effective_sample_size(weights_p) / len(weights_p):.1%})"
)
print(f"E_q[r_P] at x_obs = {np.exp(logsumexp(log_rp) - np.log(len(log_rp))):.4f}")

MU_EDGES = np.linspace(-3.5, 3.5, 71)
ALPHA_EDGES = np.linspace(-3.0, 3.0, 61)
MU_CENTERS = 0.5 * (MU_EDGES[:-1] + MU_EDGES[1:])
ALPHA_CENTERS = 0.5 * (ALPHA_EDGES[:-1] + ALPHA_EDGES[1:])
MU_CELL, ALPHA_CELL = np.meshgrid(MU_CENTERS, ALPHA_CENTERS, indexing="ij")
THETA_CELL = np.column_stack([MU_CELL.ravel(), ALPHA_CELL.ravel()])
log_truth_cell = design_logpdf(THETA_CELL) + log_likelihood(X_OBS, THETA_CELL)
truth_probability = np.exp(log_truth_cell - logsumexp(log_truth_cell)).reshape(
    MU_CELL.shape
)

def histogram_probability(theta, weights=None):
    histogram = np.histogram2d(
        theta[:, 0], theta[:, 1], bins=[MU_EDGES, ALPHA_EDGES], weights=weights
    )[0]
    return histogram / histogram.sum()

def js_distance(p, q, floor=1.0e-12):
    p = np.asarray(p, dtype=float).ravel() + floor
    q = np.asarray(q, dtype=float).ravel() + floor
    p, q = p / p.sum(), q / q.sum()
    middle = 0.5 * (p + q)
    return np.sqrt(
        0.5 * np.sum(p * np.log(p / middle))
        + 0.5 * np.sum(q * np.log(q / middle))
    )

probability_npe = histogram_probability(theta_npe)
probability_defensive = histogram_probability(theta_posterior)
probability_hnpe = histogram_probability(theta_posterior, weights_p)
distances = {
    "NPE": js_distance(truth_probability, probability_npe),
    "defensive proposal": js_distance(truth_probability, probability_defensive),
    "hNPE": js_distance(truth_probability, probability_hnpe),
}
for name, value in distances.items():
    print(f"JS distance, truth vs {name:18s}: {value:.4f}")
if distances["hNPE"] >= distances["NPE"]:
    print("Warning: this stochastic fast run did not improve JS; use FAST_MODE=False.")

fig, axes = plt.subplots(2, 2, figsize=(11, 8.5), constrained_layout=True)
levels = np.quantile(POSTERIOR_TRUTH[POSTERIOR_TRUTH > 0], [0.65, 0.85, 0.95])
axes[0, 0].contourf(MU_GRID, ALPHA_GRID, POSTERIOR_TRUTH.T, levels=25, cmap="Blues")
axes[0, 0].contour(MU_GRID, ALPHA_GRID, POSTERIOR_TRUTH.T, levels=levels, colors="navy")
axes[0, 0].set_title("analytic posterior")
for ax, probability, title in [
    (axes[0, 1], probability_npe, "spline NPE"),
    (axes[1, 0], probability_defensive, "defensive proposal"),
    (axes[1, 1], probability_hnpe, "ratio-corrected hNPE"),
]:
    ax.pcolormesh(MU_EDGES, ALPHA_EDGES, probability.T, shading="auto", cmap="Blues")
    ax.contour(MU_GRID, ALPHA_GRID, POSTERIOR_TRUTH.T, levels=levels, colors="black", linewidths=0.8)
    ax.set_title(f"{title}\nJS={distances[title if title != 'spline NPE' else 'NPE']:.3f}" if title != "ratio-corrected hNPE" else f"{title}\nJS={distances['hNPE']:.3f}")
for ax in axes.flat:
    ax.set(xlabel=r"$\mu$", ylabel=r"$\alpha$", xlim=(-3.2, 3.2), ylim=(-2.5, 2.5))
plt.show()

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(MU_GRID, TRUTH_MU, color="black", lw=2.2, label="analytic truth")
ax.hist(theta_npe[:, 0], bins=MU_EDGES, density=True, histtype="step", lw=1.8, label="NPE")
ax.hist(
    theta_posterior[:, 0], bins=MU_EDGES, weights=weights_p,
    density=True, histtype="step", lw=2.0, label="hNPE",
)
ax.set(xlabel=r"$\mu$", ylabel="posterior density", xlim=(-3.2, 3.2))
ax.legend()
ax.grid(alpha=0.25)
plt.show()


The correction should restore both the dominant positive-$\mu$ mode and the smaller negative-$\mu$ mode more accurately than the NPE alone. The defensive component is not itself the correction: it only places proposal events in regions that the ratio may need to upweight. The classifier odds determine the posterior shape.

A small ESS is not automatically a failure, but it is a warning that a few proposals dominate. If the ESS or maximum-weight diagnostic is poor, one should increase the defensive fraction, improve the NPE, or refine the simulation design before trusting tail-sensitive summaries.


## 4. Train the dual hNDE likelihood

The posterior factorization is fast but does not provide an observation generator or an absolute likelihood. The dual construction begins with an unconditional observation flow trained on the design predictive distribution,

\[
  p_\rho(x)=\int \rho(\theta)p(x\mid\theta)\,d\theta,
  \qquad q_\eta(x)\approx p_\rho(x).
\]

We never evaluate $p_\rho(x)$. Sampling it is enough: draw $\theta\sim\rho$, simulate $x\sim p(x\mid\theta)$, discard $\theta$, and train $q_\eta$ by maximum likelihood.

After freezing $q_\eta$, train a parameterized classifier with

\[
\begin{aligned}
  \ell_1(\theta,x)&=\rho(\theta)p(x\mid\theta),\\
  \ell_0(\theta,x)&=\rho(\theta)q_\eta(x).
\end{aligned}
\]

The same $\theta$ is used in both classes, while the denominator observation is independent of it. The balanced-class odds learn

\[
  r_{\rm L}(x;\theta)=\frac{p(x\mid\theta)}{q_\eta(x)},
  \qquad
  \widehat p_{\rm hNDE}(x\mid\theta)=q_\eta(x)\widehat r_{\rm L}(x;\theta).
\]


In [ ]:
rng = np.random.default_rng(SEED + 50)
theta_eta = sample_design(N_FLOW, rng)
x_eta = simulate(theta_eta, rng)
q_eta = train_spline_flow(
    x_eta,
    context=None,
    checkpoint=MODEL_DIR / "q_eta_observation_reference.pt",
    model_config=FLOW_MODEL_CONFIG,
    training_config=FLOW_TRAINING_CONFIG,
    device=device,
    seed=SEED + 51,
    load_if_available=LOAD_IF_AVAILABLE,
)
del theta_eta, x_eta
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [ ]:
rng = np.random.default_rng(SEED + 60)
theta_rl = sample_design(N_RATIO, rng)
x_rl_positive = simulate(theta_rl, rng)
x_rl_negative = sample_spline_flow(q_eta, N_RATIO)
rl_positive = np.column_stack([theta_rl, x_rl_positive])
rl_negative = np.column_stack([theta_rl, x_rl_negative])
r_l = train_ratio_classifier(
    rl_positive,
    rl_negative,
    checkpoint=MODEL_DIR / "r_l_likelihood_master.pt",
    model_config=RATIO_MODEL_CONFIG,
    training_config=RATIO_TRAINING_CONFIG,
    device=device,
    seed=SEED + 61,
    load_if_available=LOAD_IF_AVAILABLE,
)
del theta_rl, x_rl_positive, x_rl_negative, rl_positive, rl_negative
gc.collect()

# Independent likelihood-ratio calibration.
rng = np.random.default_rng(SEED + 62)
theta_cal_l = sample_design(N_CALIBRATION, rng)
x_cal_l_positive = simulate(theta_cal_l, rng)
x_cal_l_negative = sample_spline_flow(q_eta, N_CALIBRATION)
rl_cal_positive = np.column_stack([theta_cal_l, x_cal_l_positive])
rl_cal_negative = np.column_stack([theta_cal_l, x_cal_l_negative])
r_l = calibrate_ratio_classifier(r_l, rl_cal_positive, rl_cal_negative)
del theta_cal_l, x_cal_l_positive, x_cal_l_negative, rl_cal_positive, rl_cal_negative
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


### Conditional likelihood closure

At a fixed $\theta_0$, candidates from $q_\eta(x)$ can be reweighted by $r_{\rm L}(x;\theta_0)$. The result should reproduce direct simulator events at the same parameter point. The normalization identity

\[
  \mathbb E_{q_\eta}\!\left[r_{\rm L}(x;\theta_0)\right]=1
\]

is also checked. This is the observation-space analogue of the hNPE closure.


In [ ]:
THETA_CHECK = np.array([[1.2, -0.35]], dtype=np.float32)
rng = np.random.default_rng(SEED + 70)
x_reference_check = sample_spline_flow(q_eta, N_REFERENCE)
theta_reference_check = np.repeat(THETA_CHECK, len(x_reference_check), axis=0)
log_rl_check = ratio_classifier_logit(
    r_l, np.column_stack([theta_reference_check, x_reference_check])
)
weights_check = normalized_weights_from_log(log_rl_check)
x_simulator_check = simulate(
    np.repeat(THETA_CHECK, N_REFERENCE, axis=0), rng
)
print(
    "E_qeta[r_L] at theta_check =",
    np.exp(logsumexp(log_rl_check) - np.log(len(log_rl_check))),
)
print(
    f"conditional-generation ESS = {effective_sample_size(weights_check):,.0f} "
    f"/ {len(weights_check):,}"
)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.8), constrained_layout=True)
labels = [r"$x_1$", r"$x_2$", r"$x_3$"]
for index, ax in enumerate(axes):
    low, high = np.quantile(x_simulator_check[:, index], [0.002, 0.998])
    bins = np.linspace(low, high, 55)
    ax.hist(
        x_reference_check[:, index], bins=bins, density=True,
        histtype="step", color="0.6", lw=1.4, label=r"$q_\eta$ alone",
    )
    ax.hist(
        x_reference_check[:, index], bins=bins, weights=weights_check,
        density=True, histtype="step", lw=2.0, label="hNDE",
    )
    ax.hist(
        x_simulator_check[:, index], bins=bins, density=True,
        histtype="step", color="black", ls="--", lw=1.8,
        label="simulator validation",
    )
    ax.set(xlabel=labels[index], ylabel="density")
    ax.grid(alpha=0.2)
axes[0].legend(fontsize=9)
plt.show()


## 5. Verify hNPE--hNDE equivalence

Define the design-evidence correction

\[
  C_\rho(x)=\int \rho(\theta)r_{\rm L}(x;\theta)\,d\theta
  =\frac{p_\rho(x)}{q_\eta(x)}.
\]

Bayes' theorem gives the exact bridge

\[
  r_{\rm P}(\theta;x)
  =\frac{\rho(\theta)r_{\rm L}(x;\theta)}
  {C_\rho(x)q_{\phi,\epsilon}(\theta\mid x)}.
\]

Consequently the same defensive candidates can be corrected in two independent ways:

\[
\begin{aligned}
  u_j^{\rm P}&=\widehat r_{\rm P}(\theta_j;x_o),\\
  u_j^{\rm L}&=\frac{\rho(\theta_j)\widehat r_{\rm L}(x_o;\theta_j)}
  {q_{\phi,\epsilon}(\theta_j\mid x_o)}.
\end{aligned}
\]

Their normalized weights should define the same posterior. Agreement is a powerful internal diagnostic, although it cannot expose a bias shared by both networks and their common simulator.


In [ ]:
x_repeated = np.repeat(x_context, len(theta_posterior), axis=0)
log_rl_posterior = ratio_classifier_logit(
    r_l, np.column_stack([theta_posterior, x_repeated])
)
log_q_defensive = defensive_logpdf(theta_posterior, X_OBS)
log_weights_l = design_logpdf(theta_posterior) + log_rl_posterior - log_q_defensive
weights_l = normalized_weights_from_log(log_weights_l)
probability_hnde_posterior = histogram_probability(theta_posterior, weights_l)
distance_hnde = js_distance(truth_probability, probability_hnde_posterior)
distance_dual = js_distance(probability_hnpe, probability_hnde_posterior)
print(f"JS(truth, hNPE posterior) = {distances['hNPE']:.4f}")
print(f"JS(truth, hNDE posterior) = {distance_hnde:.4f}")
print(f"JS(hNPE, hNDE)            = {distance_dual:.4f}")
print(
    f"hNDE-route ESS = {effective_sample_size(weights_l):,.0f} / "
    f"{len(weights_l):,}"
)

rng = np.random.default_rng(SEED + 80)
theta_c = sample_design(N_REFERENCE, rng)
log_rl_c = ratio_classifier_logit(
    r_l, np.column_stack([theta_c, np.repeat(x_context, len(theta_c), axis=0)])
)
log_c_rho = logsumexp(log_rl_c) - np.log(len(log_rl_c))
log_bridge = (
    design_logpdf(theta_posterior)
    + log_rl_posterior
    - log_c_rho
    - log_q_defensive
)
bridge_subset = rng.choice(len(log_rp), min(20_000, len(log_rp)), replace=False)
bridge_delta = log_rp[bridge_subset] - log_bridge[bridge_subset]
print(f"C_rho(x_obs) from r_L = {np.exp(log_c_rho):.4f}")
print(f"median |log r_P - log bridge| = {np.median(np.abs(bridge_delta)):.3f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), constrained_layout=True)
axes[0].plot(MU_GRID, TRUTH_MU, color="black", lw=2.2, label="analytic truth")
axes[0].hist(
    theta_posterior[:, 0], bins=MU_EDGES, weights=weights_p,
    density=True, histtype="step", lw=2, label="posterior hNPE",
)
axes[0].hist(
    theta_posterior[:, 0], bins=MU_EDGES, weights=weights_l,
    density=True, histtype="step", lw=2, label="likelihood hNDE",
)
axes[0].set(xlabel=r"$\mu$", ylabel="posterior density", xlim=(-3.2, 3.2))
axes[0].legend()
axes[0].grid(alpha=0.25)

axes[1].hexbin(
    log_bridge[bridge_subset], log_rp[bridge_subset],
    gridsize=55, mincnt=1, bins="log", cmap="viridis",
)
low = np.quantile(np.concatenate([log_bridge[bridge_subset], log_rp[bridge_subset]]), 0.01)
high = np.quantile(np.concatenate([log_bridge[bridge_subset], log_rp[bridge_subset]]), 0.99)
axes[1].plot([low, high], [low, high], color="white", ls="--", lw=1.5)
axes[1].set(
    xlabel=r"$\log[\rho r_{\rm L}/(C_\rho q_{\phi,\epsilon})]$",
    ylabel=r"$\log r_{\rm P}$", xlim=(low, high), ylim=(low, high),
)
plt.show()


## What the dual model adds

The posterior hNPE and likelihood hNDE play different roles:

| capability | posterior hNPE | likelihood hNDE |
|---|---:|---:|
| fast parameter proposals | direct | use hNPE or another proposal |
| residual proposal correction | direct | via likelihood importance weights |
| normalized observation generator | no | yes, $q_\eta r_{\rm L}$ |
| absolute evidence | no | yes |
| alternative supported priors | importance reweighting | posterior integral |
| selection and population integrals | not by itself | direct reference expectation |

The following examples reuse the already trained networks. Simulator calls appear only in independent validation overlays explicitly labeled as such.


## 6. Posterior-predictive generation without the simulator

The posterior predictive is

\[
  p(x_{\rm rep}\mid x_o)
  =\int p(x_{\rm rep}\mid\theta)p_\rho(\theta\mid x_o)\,d\theta.
\]

We first resample $\theta_j$ from the corrected hNPE posterior. Independently draw $x_j^\star\sim q_\eta(x)$, weight each pair by $r_{\rm L}(x_j^\star;\theta_j)$, and resample. The weighted joint proposal is proportional to

\[
  p_\rho(\theta\mid x_o)q_\eta(x_{\rm rep})r_{\rm L}(x_{\rm rep};\theta)
  =p_\rho(\theta\mid x_o)p(x_{\rm rep}\mid\theta).
\]

Thus the blue sample below is generated without the original simulator. A direct simulator sample is drawn afterward solely to validate it.


In [ ]:
rng = np.random.default_rng(SEED + 90)
N_PREDICTIVE = 50_000 if FAST_MODE else 150_000
theta_predictive = theta_posterior[
    rng.choice(len(theta_posterior), size=N_PREDICTIVE, replace=True, p=weights_p)
]
x_predictive_proposal = sample_spline_flow(q_eta, N_PREDICTIVE)
log_predictive_weights = ratio_classifier_logit(
    r_l, np.column_stack([theta_predictive, x_predictive_proposal])
)
predictive_weights = normalized_weights_from_log(log_predictive_weights)
predictive_indices = rng.choice(
    N_PREDICTIVE, size=N_PREDICTIVE, replace=True, p=predictive_weights
)
x_predictive_hnde = x_predictive_proposal[predictive_indices]
print(
    f"posterior-predictive hNDE ESS = {effective_sample_size(predictive_weights):,.0f} "
    f"/ {N_PREDICTIVE:,}"
)

# Validation only: this call is not part of the simulator-free generator.
x_predictive_simulator = simulate(theta_predictive, rng)
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8), constrained_layout=True)
for index, ax in enumerate(axes):
    combined = np.concatenate(
        [x_predictive_hnde[:, index], x_predictive_simulator[:, index]]
    )
    low, high = np.quantile(combined, [0.002, 0.998])
    bins = np.linspace(low, high, 55)
    ax.hist(
        x_predictive_hnde[:, index], bins=bins, density=True,
        histtype="step", lw=2.0, label="hNPE + hNDE (no simulator)",
    )
    ax.hist(
        x_predictive_simulator[:, index], bins=bins, density=True,
        histtype="step", color="black", ls="--", lw=1.8,
        label="simulator validation",
    )
    ax.set(xlabel=rf"$x_{index + 1}^{{\rm rep}}$", ylabel="predictive density")
    ax.grid(alpha=0.2)
axes[0].legend(fontsize=9)
plt.show()


## 7. Absolute evidence

The hNDE factorization retains the absolute normalization in observation space:

\[
  p_\pi(x_o)=q_\eta(x_o)
  \int \pi(\theta)r_{\rm L}(x_o;\theta)\,d\theta.
\]

For the baseline $\pi=\rho$, the integral is $C_\rho(x_o)$. We estimate it with design samples. Unlike normalized posterior weights, the value of $q_\eta(x_o)$ must include the standardization Jacobian; the flow helper does so explicitly.

Absolute evidence is more demanding than posterior shape. A multiplicative calibration error in $r_{\rm L}$, a density-normalization error in $q_\eta$, omitted count terms, or an inconsistent selection convention directly biases it.


In [ ]:
log_q_eta_x_obs = spline_flow_log_prob(q_eta, X_OBS[None, :])[0]
rng = np.random.default_rng(SEED + 100)
N_EVIDENCE = 80_000 if FAST_MODE else 300_000
theta_evidence = sample_design(N_EVIDENCE, rng)
log_rl_evidence = ratio_classifier_logit(
    r_l,
    np.column_stack(
        [theta_evidence, np.repeat(x_context, N_EVIDENCE, axis=0)]
    ),
)
log_c_evidence = logsumexp(log_rl_evidence) - np.log(N_EVIDENCE)
log_evidence_hnde = log_q_eta_x_obs + log_c_evidence
ratio_evidence = np.exp(log_rl_evidence - np.max(log_rl_evidence))
relative_mc_error = ratio_evidence.std(ddof=1) / (
    np.sqrt(N_EVIDENCE) * ratio_evidence.mean()
)

print(f"log q_eta(x_obs)                         = {log_q_eta_x_obs: .5f}")
print(f"log C_rho(x_obs)                         = {log_c_evidence: .5f}")
print(f"log evidence from hNDE                   = {log_evidence_hnde: .5f}")
print(f"log evidence from numerical truth        = {LOG_EVIDENCE_TRUTH: .5f}")
print(f"difference (hNDE - truth)                 = {log_evidence_hnde - LOG_EVIDENCE_TRUTH: .5f}")
print(f"MC-only relative error on C_rho estimate  = {relative_mc_error:.2%}")


## 8. Nuisance-prior and auxiliary-likelihood update

The NPE and residual were trained under $\rho(\mu,\alpha)$ using only the principal observation $x$. We now replace the nuisance prior by

\[
  \pi_{\rm new}(\alpha)=\mathcal N(0.30,0.45^2)
\]

and add an auxiliary measurement

\[
  a_o=0.10,\qquad f(a_o\mid\alpha)=\mathcal N(a_o;\alpha,0.25^2).
\]

The $\mu$ prior is unchanged. Existing defensive candidates receive

\[
  u_j^{\rm new}=r_{\rm P}(\theta_j;x_o)
  \frac{\pi_{\rm new}(\alpha_j)}{\rho(\alpha_j)}
  f(a_o\mid\alpha_j).
\]

No neural model is retrained. The update is valid because the defensive design has adequate support where the new prior and auxiliary likelihood are appreciable. The ESS quantifies whether that formal support is practically useful.


In [ ]:
ALPHA_PRIOR_MEAN = 0.30
ALPHA_PRIOR_SIGMA = 0.45
A_OBSERVED = 0.10
SIGMA_A = 0.25

log_prior_update = (
    norm.logpdf(
        theta_posterior[:, 1], loc=ALPHA_PRIOR_MEAN, scale=ALPHA_PRIOR_SIGMA
    )
    - design_alpha_logpdf(theta_posterior[:, 1])
)
log_auxiliary = auxiliary_loglikelihood(
    theta_posterior[:, 1], A_OBSERVED, SIGMA_A
)
log_weights_update = log_rp + log_prior_update + log_auxiliary
weights_update = normalized_weights_from_log(log_weights_update)
print(
    f"updated-posterior ESS = {effective_sample_size(weights_update):,.0f} / "
    f"{len(weights_update):,} "
    f"({effective_sample_size(weights_update) / len(weights_update):.1%})"
)

log_joint_update = (
    design_mu_logpdf(THETA_GRID[:, 0])
    + norm.logpdf(
        THETA_GRID[:, 1], loc=ALPHA_PRIOR_MEAN, scale=ALPHA_PRIOR_SIGMA
    )
    + log_likelihood(X_OBS, THETA_GRID)
    + auxiliary_loglikelihood(THETA_GRID[:, 1], A_OBSERVED, SIGMA_A)
).reshape(MU_MESH.shape)
update_shift = log_joint_update.max()
update_unnormalized = np.exp(log_joint_update - update_shift)
update_integral = np.trapezoid(
    np.trapezoid(update_unnormalized, ALPHA_GRID, axis=1), MU_GRID
)
posterior_update_truth = update_unnormalized / update_integral
update_mu_truth = np.trapezoid(posterior_update_truth, ALPHA_GRID, axis=1)
update_alpha_truth = np.trapezoid(posterior_update_truth, MU_GRID, axis=0)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), constrained_layout=True)
axes[0].plot(MU_GRID, update_mu_truth, color="black", lw=2.2, label="updated analytic truth")
axes[0].hist(
    theta_posterior[:, 0], bins=MU_EDGES, weights=weights_update,
    density=True, histtype="step", lw=2.0, label="updated hNPE",
)
axes[0].plot(MU_GRID, TRUTH_MU, color="0.6", ls="--", label="baseline truth")
axes[0].set(xlabel=r"$\mu$", ylabel="posterior density", xlim=(-3.2, 3.2))
axes[1].plot(
    ALPHA_GRID, update_alpha_truth, color="black", lw=2.2,
    label="updated analytic truth",
)
axes[1].hist(
    theta_posterior[:, 1], bins=ALPHA_EDGES, weights=weights_update,
    density=True, histtype="step", lw=2.0, label="updated hNPE",
)
axes[1].plot(ALPHA_GRID, TRUTH_ALPHA, color="0.6", ls="--", label="baseline truth")
axes[1].set(xlabel=r"$\alpha$", ylabel="posterior density", xlim=(-2.5, 2.5))
for ax in axes:
    ax.legend(fontsize=9)
    ax.grid(alpha=0.25)
plt.show()


## 9. Selection integrals without the simulator

Suppose an event is retained when

\[
  I_{\rm sel}(x)=\mathbf 1[x_1>0.5]\,\mathbf 1[x_3>0].
\]

At fixed parameters the efficiency is

\[
\begin{aligned}
  \beta(\mu,\alpha)
  &=\int I_{\rm sel}(x)p(x\mid\mu,\alpha)\,dx\\
  &=\mathbb E_{x\sim q_\eta}
  \left[I_{\rm sel}(x)r_{\rm L}(x;\mu,\alpha)\right].
\end{aligned}
\]

One renewable sample from $q_\eta$ can therefore be reused for every parameter point. We use the self-normalized finite-sample form

\[
  \widehat\beta=\frac{\sum_i I_{\rm sel}(x_i)\widehat r_{\rm L}(x_i;\theta)}
  {\sum_i\widehat r_{\rm L}(x_i;\theta)},
\]

whose denominator should approach one and removes a finite classifier normalization offset at each $\theta$. Because $x_1$ and $x_3$ are conditionally independent Gaussians here, an analytic answer is available for validation.


In [ ]:
X1_THRESHOLD = 0.5
X3_THRESHOLD = 0.0
N_SELECTION_REFERENCE = 25_000 if FAST_MODE else 80_000
MU_SELECTION = np.linspace(-2.4, 2.4, 33)
ALPHA_SELECTION = [-0.7, 0.0, 0.7]
x_selection_reference = sample_spline_flow(q_eta, N_SELECTION_REFERENCE)
selection_indicator = (
    (x_selection_reference[:, 0] > X1_THRESHOLD)
    & (x_selection_reference[:, 2] > X3_THRESHOLD)
)

def exact_selection_probability(theta):
    mean = simulator_mean(theta)
    return norm.sf(
        (X1_THRESHOLD - mean[:, 0]) / SIMULATOR_SIGMA[0]
    ) * norm.sf(
        (X3_THRESHOLD - mean[:, 2]) / SIMULATOR_SIGMA[2]
    )


selection_results = {}
normalization_results = {}
for alpha in ALPHA_SELECTION:
    estimates, normalizations = [], []
    for mu in MU_SELECTION:
        theta_value = np.array([[mu, alpha]], dtype=np.float32)
        theta_repeated = np.repeat(
            theta_value, N_SELECTION_REFERENCE, axis=0
        )
        log_ratio = ratio_classifier_logit(
            r_l, np.column_stack([theta_repeated, x_selection_reference])
        )
        ratio = np.exp(log_ratio - np.max(log_ratio))
        estimates.append(np.sum(selection_indicator * ratio) / np.sum(ratio))
        # Recover the unshifted sample mean stably for the normalization check.
        normalizations.append(np.exp(logsumexp(log_ratio) - np.log(len(log_ratio))))
    selection_results[alpha] = np.asarray(estimates)
    normalization_results[alpha] = np.asarray(normalizations)

fig, ax = plt.subplots(figsize=(7.3, 4.8))
colors = ["C0", "C1", "C2"]
for color, alpha in zip(colors, ALPHA_SELECTION):
    theta_line = np.column_stack(
        [MU_SELECTION, np.full_like(MU_SELECTION, alpha)]
    )
    exact = exact_selection_probability(theta_line)
    ax.plot(
        MU_SELECTION, exact, color=color, ls="--", lw=2,
        label=rf"analytic, $\alpha={alpha:+.1f}$",
    )
    ax.plot(
        MU_SELECTION, selection_results[alpha], color=color, marker="o",
        ms=3, lw=1.2, label=rf"hNDE, $\alpha={alpha:+.1f}$",
    )
    max_norm_error = np.max(np.abs(normalization_results[alpha] - 1.0))
    print(f"alpha={alpha:+.1f}: max |E_q[r_L]-1| = {max_norm_error:.3f}")
ax.set(
    xlabel=r"$\mu$", ylabel=r"selection probability $\beta(\mu,\alpha)$",
    ylim=(-0.02, 1.02),
)
ax.legend(ncol=2, fontsize=9)
ax.grid(alpha=0.25)
plt.show()


## Conclusions and limitations

This exercise built two complementary hybrid factorizations:

\[
  p_\rho(\theta\mid x)
  =q_{\phi,\epsilon}(\theta\mid x)r_{\rm P}(\theta;x),
  \qquad
  p(x\mid\theta)=q_\eta(x)r_{\rm L}(x;\theta).
\]

The first turns a quadratic-spline NPE into a correctable, diagnosable importance proposal. The second supplies a normalized and generative likelihood surrogate. Their bridge relation provides an internal consistency test, and their combination enables calculations that a posterior flow alone cannot perform.

Four simulator-free uses were demonstrated after training:

1. posterior-predictive generation from corrected posterior parameters and hNDE observation proposals;
2. absolute evidence from $q_\eta(x_o)\,\mathbb E_\pi[r_{\rm L}(x_o;\theta)]$;
3. a nuisance-prior and Gaussian auxiliary update by importance reweighting;
4. selection integrals from a renewable $q_\eta$ reference sample.

The word *corrected* must be interpreted carefully. The ratios correct the flows relative to the learned classifier targets and the training simulator. They cannot repair simulator misspecification, finite-sample classifier bias, or missing support. Agreement between hNPE and hNDE is necessary but not sufficient because the two models may share data and representation biases. Independent simulator validation, simulation-based calibration, coverage/rank diagnostics, ESS and tail checks, and explicit likelihood-normalization tests remain essential.

### Suggested investigations

1. Set DEFENSIVE_EPSILON to zero and determine whether the negative-$\mu$ mode can still be recovered.
2. Increase the NPE capacity while keeping the ratio fixed. Does $r_{\rm P}$ approach unity and does the ESS improve?
3. Move the nuisance prior beyond the broad design component and observe how the ESS warns before the update fails.
4. Replace the Gaussian auxiliary likelihood by a Student-$t$ calibration and repeat the update without retraining.
5. Use a selection concentrated in the far tail. Compare plain $q_\eta$ sampling with an importance proposal targeted near the selection boundary.
6. Repeat the entire exercise over many $x_o\sim p_\rho(x)$ and perform simulation-based calibration for the uncorrected NPE, hNPE, and hNDE-route posterior.
